# Step 5 Preprocess and Split Notebook

This notebook duplicates the Step 5 preprocessing and split logic in a self-contained format for Google Colab. It reads the Step 4 raw sentence dataset from Google Drive, reads the Step 5 locked holdout source from Google Drive, cleans and deduplicates sentences, recreates the holdout, creates train/test outputs, and writes all generated files back to the Step 5 Google Drive folder.

Run the notebook from top to bottom after Step 4 has created:

- `Team 8 - Capstone Project/Step 4 - Dataset Collection/data/raw_dataset.csv`

Step 5 also needs this locked source file in Drive:

- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/original_holdout_1000.csv`

Generated outputs are written to:

- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/holdout_1000.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/train_test_15000.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/train.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/test.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/preprocess_split_summary.txt`

## 1. Imports, Drive Setup, and Config

This section imports the libraries used by the Step 5 script, mounts Google Drive, defines Drive paths, and sets the same constants, sampling settings, industry mappings, and output columns.

In [ ]:
from importlib import import_module
from pathlib import Path

import pandas as pd

# Mount Google Drive so Step 4 input and Step 5 outputs persist for everyone.
try:
    drive = import_module("google.colab").drive
except ModuleNotFoundError as exc:
    raise RuntimeError("Run this notebook in Google Colab so Google Drive can be mounted.") from exc

drive.mount("/content/drive")

RANDOM_SEED = 42
HOLDOUT_SIZE = 1000
TRAIN_TEST_SIZE = 15000
TEST_FRACTION = 0.20

# Make sure these exactly match the shared Drive folder structure.
PROJECT_DRIVE_BASE = Path("/content/drive/MyDrive/Team 8 - Capstone Project")
STEP4_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 4 - Dataset Collection"
STEP5_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 5 - Preprocess Split"

STEP4_DATA_DIR = STEP4_DRIVE_BASE / "data"
DATA_DIR = STEP5_DRIVE_BASE / "data"

# Step 5 works directly from the Step 4 Drive output.
INPUT_FILE = STEP4_DATA_DIR / "raw_dataset.csv"
LOCKED_HOLDOUT_FILE = DATA_DIR / "original_holdout_1000.csv"
HOLDOUT_FILE = DATA_DIR / "holdout_1000.csv"
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"
TRAIN_TEST_FILE = DATA_DIR / "train_test_15000.csv"
SUMMARY_FILE = DATA_DIR / "preprocess_split_summary.txt"

INDUSTRY_TICKERS = {
    "Tech": ["AAPL", "MSFT", "GOOGL", "META", "NVDA", "INTC", "IBM", "ORCL", "CSCO", "ADBE"],
    "Healthcare": ["JNJ", "PFE", "MRK", "ABT", "BMY", "AMGN", "GILD", "MDT", "UNH", "CVS"],
    "Finance": ["JPM", "BAC", "WFC", "GS", "MS", "C", "AXP", "BLK", "COF", "USB"],
    "Consumer/Retail": ["WMT", "AMZN", "TGT", "COST", "HD", "LOW", "NKE", "SBUX", "MCD", "YUM"],
    "Energy": ["XOM", "CVX", "COP", "SLB", "PSX", "VLO", "MPC", "OXY", "HES", "DVN"],
    "Industrials": ["GE", "HON", "MMM", "CAT", "DE", "BA", "LMT", "RTX", "UPS", "FDX"],
    "Telecom/Media": ["T", "VZ", "CMCSA", "DIS", "NFLX", "PARA", "WBD", "FOXA", "DISH", "LUMN"],
    "Materials": ["DD", "DOW", "LIN", "APD", "NEM", "FCX", "VMC", "MLM", "PKG", "IP"],
}

TICKER_TO_INDUSTRY = {
    ticker: industry
    for industry, tickers in INDUSTRY_TICKERS.items()
    for ticker in tickers
}

OUTPUT_COLUMNS = ["sentence_id", "sentence", "ticker", "industry", "filing_date", "filing_id"]

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Step 4 input file: {INPUT_FILE}")
print(f"Step 5 data directory: {DATA_DIR}")

## 2. Input Files

Step 5 reads Step 4's generated `raw_dataset.csv` directly from the Step 4 Google Drive `data/` folder. It also expects the locked holdout source to already be in the Step 5 Google Drive `data/` folder.

In [ ]:
def verify_drive_inputs():
    """Confirm required Drive inputs exist before running the split."""
    missing = []
    if not INPUT_FILE.exists():
        missing.append(
            f"Step 4 raw dataset not found: {INPUT_FILE}\n"
            "Run the Step 4 notebook first, or confirm the Step 4 Drive folder name is correct."
        )
    if not LOCKED_HOLDOUT_FILE.exists():
        missing.append(
            f"Locked holdout source not found: {LOCKED_HOLDOUT_FILE}\n"
            "Upload original_holdout_1000.csv to the Step 5 Drive data folder before running."
        )

    if missing:
        raise FileNotFoundError("\n\n".join(missing))

    print(f"Step 4 raw input exists: {INPUT_FILE}")
    print(f"Locked holdout exists: {LOCKED_HOLDOUT_FILE}")

    # Quick test to confirm the Step 5 Drive output folder is writable.
    test_file = DATA_DIR / "test_file.csv"
    pd.DataFrame({"check": [1, 2, 3]}).to_csv(test_file, index=False)
    print(f"Step 5 Drive write test succeeded: {test_file}")


verify_drive_inputs()

## 3. Load and Clean Sentences

This section duplicates the Step 5 cleaning behavior: read the raw sentences, remove blank and duplicate text, map tickers to industries, add filing year, and drop unknown industries.

In [ ]:
def add_filing_year(df):
    df = df.copy()
    df["filing_year"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.year.astype("Int64").astype(str)
    return df


def load_sentences():
    df = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig")
    df = df.dropna(subset=["sentence"])
    df["sentence"] = df["sentence"].str.strip()
    df = df[df["sentence"].ne("")]
    df = df.drop_duplicates(subset=["sentence"]).copy()
    df["ticker"] = df["ticker"].str.upper()
    df["industry"] = df["ticker"].map(TICKER_TO_INDUSTRY).fillna("Unknown")
    df = add_filing_year(df)
    return df[df["industry"] != "Unknown"].copy()


cleaned_sentences = load_sentences()
print(f"Raw cleaned sentences available: {len(cleaned_sentences):,}")
cleaned_sentences.head(5)

## 4. Locked Holdout Handling

The original locked holdout is read from the Step 5 Google Drive `data/` folder and validated, but never overwritten. The generated `holdout_1000.csv` is written separately into the same Step 5 Drive output folder.

In [ ]:
def load_locked_holdout():
    """Read the hand-labeled holdout source without modifying it."""
    if not LOCKED_HOLDOUT_FILE.exists():
        return None

    holdout = pd.read_csv(LOCKED_HOLDOUT_FILE, dtype=str, encoding="utf-8-sig")
    missing_columns = sorted(set(OUTPUT_COLUMNS) - set(holdout.columns))
    if missing_columns:
        raise ValueError(
            f"{LOCKED_HOLDOUT_FILE} is missing required columns: {', '.join(missing_columns)}"
        )

    holdout = holdout[OUTPUT_COLUMNS].dropna(subset=["sentence"]).copy()
    holdout["sentence"] = holdout["sentence"].str.strip()
    holdout = holdout[holdout["sentence"].ne("")]

    if len(holdout) != HOLDOUT_SIZE:
        raise ValueError(
            f"{LOCKED_HOLDOUT_FILE} should contain {HOLDOUT_SIZE} rows, found {len(holdout)}"
        )
    if holdout["sentence_id"].duplicated().any():
        raise ValueError(f"{LOCKED_HOLDOUT_FILE} contains duplicate sentence_id values")
    if holdout["sentence"].duplicated().any():
        raise ValueError(f"{LOCKED_HOLDOUT_FILE} contains duplicate sentence values")

    return add_filing_year(holdout)


def remove_holdout_from_pool(df, holdout):
    remaining = df.copy()

    if "sentence_id" in remaining.columns and "sentence_id" in holdout.columns:
        remaining = remaining[~remaining["sentence_id"].isin(holdout["sentence_id"])]

    return remaining[~remaining["sentence"].isin(holdout["sentence"])].copy()


def write_generated_csv(df, output_file):
    if output_file.resolve() == LOCKED_HOLDOUT_FILE.resolve():
        raise ValueError(f"Refusing to overwrite locked holdout source: {LOCKED_HOLDOUT_FILE}")

    df[OUTPUT_COLUMNS].to_csv(output_file, index=False)

## 5. Balanced Sampling Helpers

These functions spread selected rows across industry, ticker, and filing year where possible, matching the existing Step 5 script.

In [ ]:
def allocate_targets(group_sizes, total_target):
    """Allocate target rows across groups, redistributing shortages to groups with capacity."""
    groups = list(group_sizes.index)
    if not groups:
        return {}

    base = total_target // len(groups)
    remainder = total_target % len(groups)
    targets = {
        group: min(int(group_sizes[group]), base + (1 if i < remainder else 0))
        for i, group in enumerate(groups)
    }

    while sum(targets.values()) < min(total_target, int(group_sizes.sum())):
        available = {
            group: int(group_sizes[group]) - targets[group]
            for group in groups
            if targets[group] < int(group_sizes[group])
        }
        if not available:
            break

        total_available = sum(available.values())
        remaining = min(total_target, int(group_sizes.sum())) - sum(targets.values())

        for group in sorted(available, key=available.get, reverse=True):
            if remaining <= 0:
                break
            add = max(1, round(remaining * available[group] / total_available))
            add = min(add, available[group], remaining)
            targets[group] += add
            remaining -= add

    return targets


def sample_evenly(df, group_columns, target):
    """Sample target rows while spreading picks evenly across nested groups."""
    if target <= 0 or df.empty:
        return df.head(0)
    if len(df) <= target:
        return df.copy()
    if not group_columns:
        return df.sample(n=target, random_state=RANDOM_SEED)

    group_sizes = df.groupby(group_columns, dropna=False).size()
    targets = allocate_targets(group_sizes, target)

    samples = []
    for group_key, group_target in targets.items():
        if group_target <= 0:
            continue
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        mask = pd.Series(True, index=df.index)
        for column, value in zip(group_columns, group_key):
            mask &= df[column].eq(value)

        group_df = df[mask]
        samples.append(sample_evenly(group_df, group_columns[1:], group_target))

    sampled = pd.concat(samples) if samples else df.head(0)
    if len(sampled) > target:
        sampled = sampled.sample(n=target, random_state=RANDOM_SEED)
    return sampled


def sample_pool(df, target):
    """Sample up to target rows across industry, ticker, and filing year."""
    if len(df) <= target:
        return df.copy()

    parts = []
    industry_targets = allocate_targets(df.groupby("industry").size(), target)
    for industry, industry_target in industry_targets.items():
        industry_df = df[df["industry"] == industry]
        parts.append(sample_evenly(industry_df, ["ticker", "filing_year"], industry_target))

    sampled = pd.concat(parts).drop_duplicates(subset=["sentence"])
    if len(sampled) > target:
        sampled = sampled.sample(n=target, random_state=RANDOM_SEED)
    return sampled

## 6. Holdout, Train/Test Split, and Summary

This section duplicates the final dataset assembly, length statistics, imbalance plan, and summary text generation.

In [ ]:
def create_sampled_holdout(df):
    holdout = sample_pool(df, HOLDOUT_SIZE)

    remaining_needed = min(HOLDOUT_SIZE, len(df)) - len(holdout)
    if remaining_needed > 0:
        remaining_pool = df.drop(index=holdout.index)
        holdout = pd.concat([
            holdout,
            sample_evenly(remaining_pool, ["industry", "ticker", "filing_year"], remaining_needed),
        ])

    return holdout


def get_holdout(df):
    locked_holdout = load_locked_holdout()
    if locked_holdout is not None:
        return locked_holdout

    return create_sampled_holdout(df)


def split_train_test(train_test):
    test_size = round(len(train_test) * TEST_FRACTION)
    test = sample_pool(train_test, test_size)
    train = train_test.drop(index=test.index)
    return train, test


def length_stats(df):
    lengths = df["sentence"].str.split().str.len()
    return {
        "rows": len(df),
        "mean_words": round(lengths.mean(), 2),
        "median_words": round(lengths.median(), 2),
        "min_words": int(lengths.min()),
        "max_words": int(lengths.max()),
    }


def format_stats(name, df):
    stats = length_stats(df)
    return (
        f"{name}: rows={stats['rows']}, mean_words={stats['mean_words']}, "
        f"median_words={stats['median_words']}, min_words={stats['min_words']}, "
        f"max_words={stats['max_words']}"
    )


def build_summary(df, holdout, train_test, train, test):
    train_test_shortfall = max(0, TRAIN_TEST_SIZE - len(train_test))
    lines = [
        "Step 5 Preprocess and Split Summary",
        "=" * 36,
        f"Raw cleaned sentences available: {len(df)}",
        f"Locked holdout sentences: {len(holdout)}",
        f"Train/test target sentences: {TRAIN_TEST_SIZE}",
        f"Train/test actual sentences: {len(train_test)}",
        f"Train/test shortfall sentences: {train_test_shortfall}",
        f"Train sentences: {len(train)}",
        f"Test sentences: {len(test)}",
        "",
        "Length statistics:",
        format_stats("All cleaned", df),
        format_stats("Holdout", holdout),
        format_stats("Train", train),
        format_stats("Test", test),
        "",
        "Holdout breakdown by industry:",
        holdout["industry"].value_counts().sort_index().to_string(),
        "",
        "Train/test breakdown by industry:",
        train_test["industry"].value_counts().sort_index().to_string(),
        "",
        "Holdout breakdown by company:",
        holdout["ticker"].value_counts().sort_index().to_string(),
        "",
        "Holdout breakdown by filing year:",
        holdout["filing_year"].value_counts().sort_index().to_string(),
        "",
        "Imbalance plan:",
        "The holdout is fixed from the hand-labeled original so labels stay reproducible. The train/test files are sampled across industry, ticker, and filing year to reduce firm concentration. After human labels are available, report label distributions and use macro F1/MCC plus class weighting or balanced sampling if one strategic-orientation class is rare.",
        "",
    ]
    return "\n".join(lines)

## 7. Run Preprocessing and Save Outputs

The next cell runs the full Step 5 pipeline and saves all generated CSVs and the summary text file into `data/` next to this notebook.

In [ ]:
def run_preprocess_split():
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    df = load_sentences()
    holdout = get_holdout(df)
    remaining = remove_holdout_from_pool(df, holdout)
    train_test = sample_pool(remaining, TRAIN_TEST_SIZE)
    train, test = split_train_test(train_test)

    write_generated_csv(holdout, HOLDOUT_FILE)
    write_generated_csv(train_test, TRAIN_TEST_FILE)
    write_generated_csv(train, TRAIN_FILE)
    write_generated_csv(test, TEST_FILE)

    summary = build_summary(df, holdout, train_test, train, test)
    with SUMMARY_FILE.open("w", encoding="utf-8") as summary_file:
        summary_file.write(summary)

    print(summary)
    print(f"Saved holdout to: {HOLDOUT_FILE}")
    print(f"Saved train/test set to: {TRAIN_TEST_FILE}")
    print(f"Saved train split to: {TRAIN_FILE}")
    print(f"Saved test split to: {TEST_FILE}")
    print(f"Saved summary to: {SUMMARY_FILE}")

    return df, holdout, train_test, train, test, summary


df, holdout, train_test, train, test, summary = run_preprocess_split()

## 8. Sample Outputs and Checks

These cells show the first few rows and basic counts so another person can confirm they reproduced the expected Step 5 outputs.

In [ ]:
print("Generated row counts")
print(f"Holdout: {len(holdout):,}")
print(f"Train/test pool: {len(train_test):,}")
print(f"Train: {len(train):,}")
print(f"Test: {len(test):,}")

print("\nGenerated files")
for output_file in [HOLDOUT_FILE, TRAIN_TEST_FILE, TRAIN_FILE, TEST_FILE, SUMMARY_FILE]:
    print(f"{output_file}: {output_file.exists()}")

train_test.head(5)

In [ ]:
# Show the generated summary text exactly as it was saved.
print(SUMMARY_FILE.read_text(encoding="utf-8"))

## Notes on Reproducibility and Limitations

- The split is deterministic because all random sampling uses `RANDOM_SEED = 42`.
- The notebook reads Step 4's `raw_dataset.csv` directly from the Step 4 Google Drive `data/` folder.
- The locked holdout source is validated from the Step 5 Google Drive `data/` folder, but it is not overwritten by generated outputs.
- To reproduce the same row counts, use the same Step 4 `raw_dataset.csv` and the same `original_holdout_1000.csv`.
- The train/test pool targets 15,000 rows, but the actual count can be lower when the cleaned raw dataset does not contain enough unique sentences after removing the locked holdout.
- The balancing logic spreads samples across industry, ticker, and filing year where possible, but it cannot create rows for groups that are underrepresented in the raw data.